In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import io
import re
import json
import os

print("Sandbox workspace environments successfully initiated.")

In [1]:
def extract_raw_text(file_path):
    """Detects document types and returns unformatted plain string buffers."""
    if not os.path.exists(file_path):
        return f"Error: Target document path not found: {file_path}"
        
    ext = file_path.split(".")[-1].lower()
    text = ""
    
    if ext == "pdf":
        doc = fitz.open(file_path)
        for page in doc:
            text += page.get_text()
        
        # Fallback to OCR if the PDF is just a scanned image wrapper
        if not text.strip():
            print("Readable string tokens missing. Engaging PyMuPDF OCR fallbacks...")
            for page in doc:
                pix = page.get_pixmap()
                image_data = pix.tobytes("png")
                image = Image.open(io.BytesIO(image_data))
                text += pytesseract.image_to_string(image)
    else:
        # Direct Image handler pathing
        image = Image.open(file_path)
        text = pytesseract.image_to_string(image)
        
    return text

print("Extraction algorithm successfully compiled.")

Extraction algorithm successfully compiled.


In [ ]:
# Mock mock text resembling standard university result cards for sandbox prototyping
sample_ocr_output = """
GLOBAL ACADEMIC TECHNICAL UNIVERSITY
NAME: Active Student | TERM: SEMESTER 3
---------------------------------------------
SUBJECT CODE | SUBJECT NAME          | MARKS | MAX | GRADE
CS301        | DATA STRUCTURES       | 42    | 100 | E
CS302        | SYSTEM ARCHITECTURE   | 58    | 100 | D
MA303        | DISCRETE MATHEMATICS  | 88    | 100 | A
---------------------------------------------
"""

def mock_parse_to_json(raw_text):
    """Regex pipeline to pull structured records out of raw text blocks."""
    records = []
    # Dynamic parsing regex to isolate code strings, naming blocks, and marks
    pattern = re.compile(r"([A-Z]{2}\d{3})\s*\|\s*([A-Za-z\s]+?)\s*\|\s*(\d+)\s*\|\s*(\d+)\s*\|")
    
    matches = pattern.findall(raw_text)
    for match in matches:
        code, name, marks, max_m = match
        marks = float(marks)
        max_m = float(max_m)
        percentage = (marks / max_m) * 100
        
        records.append({
            "semester": 3,
            "subject_code": code.strip(),
            "subject_name": name.strip(),
            "marks_obtained": marks,
            "max_marks": max_m,
            "percentage": percentage,
            "is_failed": marks < 50,
            "needs_focus": percentage < 60
        })
    return records

test_data = mock_parse_to_json(sample_ocr_output)
print(json.dumps(test_data, indent=4))
